Import potrzebnych bibliotek

In [303]:
import pandas as pd
import numpy as np
import time

Wczytanie danych

In [304]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


In [305]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ARTIST              22110 non-null  int64  
 1   TECHNIQUE           22110 non-null  float64
 2   SIGNATURE           22110 non-null  int64  
 3   CONDITION           22110 non-null  int64  
 4   TOTAL DIMENSIONS    22110 non-null  float64
 5   YEAR                22110 non-null  float64
 6   Colorfulness Score  22110 non-null  float64
 7   SVD Entropy         22110 non-null  float64
 8   PRICE               22110 non-null  int64  
dtypes: float64(5), int64(4)
memory usage: 1.5 MB


Dopasowanie typów zmiennych

In [306]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']

bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)


Dane treningowe i testowe

In [307]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [308]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [309]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)


In [ ]:
# --- PRZEŁĄCZNIK METODY ---
# Ustaw na False, by użyć starej metody (Z-score). 
# Ustaw na True, by użyć nowej metody (Transformacja logarytmiczna).
UZYJ_LOGARYTMU = True
# --------------------------

if UZYJ_LOGARYTMU:
    print("Używamy transformacji logarytmicznej (zapobiega ujemnym cenom).")
    
    y_train = np.log(df_train['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test = np.log(df_test['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    
    
    def prawdziwa_cena(wyliczona_wartosc):
        return np.exp(wyliczona_wartosc)
        
else:
    print("Używamy normalizacji Z-score (może generować ujemne ceny).")
    
    srednia_cena = df_train['PRICE'].mean()
    odchylenie_cena = df_train['PRICE'].std()
    
    y_train = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
    
    
    def prawdziwa_cena(znormalizowana_cena):
        return znormalizowana_cena * odchylenie_cena + srednia_cena

Używamy transformacji logarytmicznej (zapobiega ujemnym cenom).


Tworzenie Dense Layer

In [311]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases
        
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)

Funkcje aktywacji

In [312]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

class Activation_Linear:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs
    
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()

Funkcja straty

In [313]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss
    
class Loss_MSE(Loss):
    def forward(self, y_pred, y_true):
        sample_losses = np.mean((y_pred - y_true) ** 2, axis=-1)
        return sample_losses
    
    def backward(self, dvalues, y_true):
        liczba_probek = len(dvalues)
        self.dinputs = -2 * (y_true - dvalues) / liczba_probek



Optymalizator SGD

In [314]:
class Optimizer_SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update_params(self, layer):
        layer.weights -= self.learning_rate * layer.dweights
        layer.biases -= self.learning_rate * layer.dbiases

Model Bazowy

In [ ]:

# Domyślne parametry
n1, n2 = 64, 32
domyslny_lr = 0.1
domyslny_batch = 256
epoki = 100

dense1_b = Layer_Dense(X_train.shape[1], n1)
activation1_b = Activation_ReLU()

dense2_b = Layer_Dense(n1, n2)
activation2_b = Activation_ReLU()

dense3_b = Layer_Dense(n2, 1)
activation3_b = Activation_Linear()

loss_function_b = Loss_MSE()
optimizer_b = Optimizer_SGD(learning_rate=domyslny_lr)

# Trening
for epoch in range(epoki):
    for start_idx in range(0, len(X_train), domyslny_batch):
        end_idx = start_idx + domyslny_batch
        X_batch = X_train[start_idx:end_idx]
        y_batch = y_train[start_idx:end_idx]
        
        dense1_b.forward(X_batch)
        activation1_b.forward(dense1_b.output)
        dense2_b.forward(activation1_b.output)
        activation2_b.forward(dense2_b.output)
        dense3_b.forward(activation2_b.output)
        activation3_b.forward(dense3_b.output)
        
        loss_function_b.backward(activation3_b.output, y_batch)
        activation3_b.backward(loss_function_b.dinputs)
        dense3_b.backward(activation3_b.dinputs)
        activation2_b.backward(dense3_b.dinputs)
        dense2_b.backward(activation2_b.dinputs)
        activation1_b.backward(dense2_b.dinputs)
        dense1_b.backward(activation1_b.dinputs)
        
        optimizer_b.update_params(dense1_b)
        optimizer_b.update_params(dense2_b)
        optimizer_b.update_params(dense3_b)

# Ewaluacja
dense1_b.forward(X_test)
activation1_b.forward(dense1_b.output)
dense2_b.forward(activation1_b.output)
activation2_b.forward(dense2_b.output)
dense3_b.forward(activation2_b.output)
activation3_b.forward(dense3_b.output)

wymyslone_ceny_dolary_b = prawdziwa_cena(activation3_b.output)
prawdziwe_ceny_dolary_b = prawdziwa_cena(y_test)

mae_b = np.mean(np.abs(prawdziwe_ceny_dolary_b - wymyslone_ceny_dolary_b))
rmse_b = np.sqrt(np.mean((prawdziwe_ceny_dolary_b - wymyslone_ceny_dolary_b)**2))
ss_res_b = np.sum((prawdziwe_ceny_dolary_b - wymyslone_ceny_dolary_b)**2)
ss_tot_b = np.sum((prawdziwe_ceny_dolary_b - np.mean(prawdziwe_ceny_dolary_b))**2)
r2_b = 1 - (ss_res_b / ss_tot_b)

non_zero_b = prawdziwe_ceny_dolary_b != 0
mape_b = np.mean(np.abs((prawdziwe_ceny_dolary_b[non_zero_b] - wymyslone_ceny_dolary_b[non_zero_b]) / prawdziwe_ceny_dolary_b[non_zero_b])) * 100

print("\n" + "="*50)
print("RAPORT STARTOWY: MODEL BAZOWY (DANE TESTOWE)")
print("="*50)
print(f"Przetwarzanie danych: Normalizacja Z-Score")
print(f"Parametry: [64, 32], LR=0.1, Batch=256")
print("-" * 50)
print(f"1. MAE (Średnia pomyłka):     {mae_b:10.2f} $")
print(f"2. RMSE (Kara za ekstrema):   {rmse_b:10.2f} $")
print(f"3. Współczynnik R^2:          {r2_b:10.4f} (max 1.0)")
print(f"4. MAPE (Błąd procentowy):    {mape_b:10.2f} %")
print("="*50)

Trenowanie modelu bazowego (Z-Score, domyślne parametry)...

RAPORT STARTOWY: MODEL BAZOWY (DANE TESTOWE)
Przetwarzanie danych: Normalizacja Z-Score
Parametry: [64, 32], LR=0.1, Batch=256
--------------------------------------------------
1. MAE (Średnia pomyłka):         113.29 $
2. RMSE (Kara za ekstrema):       356.09 $
3. Współczynnik R^2:              0.4268 (max 1.0)
4. MAPE (Błąd procentowy):        161.63 %


Badanie liczby neuronów

In [ ]:

# 4 różne warianty liczby neuronów (warstwa1, warstwa2)
architektury_do_testu = [
    (16, 8),
    (32, 16),
    (64, 32),
    (128, 64),
    (256, 128)
]
powtorzenia = 3       
epoki = 100           
batch_size = 256
learning_rate = 0.1  

wyniki_raport = []
liczba_cech = X_train.shape[1]

print("="*60)
print("ROZPOCZYNAMY BADANIE: Rozmiar warstw ukrytych")
print("="*60)

for arch in architektury_do_testu:
    n1, n2 = arch
    print(f"\n---> Testuję architekturę: [{n1}, {n2}] Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=learning_rate)
        
        
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), batch_size):
                end_idx = start_idx + batch_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    
    wyniki_raport.append({
        "Architektura": f"[{n1}, {n2}]",
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2),
        "Najlepsze MAE (Test) [$]": round(np.min(mae_test_historia), 2)
    })


print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW ARCHITEKTURY")
print("="*60)
df_raport = pd.DataFrame(wyniki_raport)
print(df_raport.to_string(index=False))

Badanie Learning rate

In [ ]:

n1, n2 = 128, 64

# 4 warianty współczynnika uczenia do przetestowania:
learning_rates_do_testu = [0.1, 0.05, 0.01, 0.001]

powtorzenia = 3       
epoki = 100           
batch_size = 256      

wyniki_raport_lr = []
liczba_cech = X_train.shape[1]

print("="*60)
print(f"ROZPOCZYNAMY BADANIE: Współczynnik uczenia Sieć: [{n1}, {n2}])")
print("="*60)

for lr in learning_rates_do_testu:
    print(f"\n---> Testuję Learning Rate: {lr} Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        
        optimizer = Optimizer_SGD(learning_rate=lr) 
        
        
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), batch_size):
                end_idx = start_idx + batch_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")


    wyniki_raport_lr.append({
        "Learning Rate": lr,
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2),
        "Najlepsze MAE (Test) [$]": round(np.min(mae_test_historia), 2)
    })

print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW WSPÓŁCZYNNIKA UCZENIA (LR)")
print("="*60)
df_raport_lr = pd.DataFrame(wyniki_raport_lr)
print(df_raport_lr.to_string(index=False))

Badanie Batch Size

In [ ]:

n1, n2 = 128, 64
najlepszy_lr = 0.05

# 4 warianty wielkości paczki do przetestowania:
batch_sizes_do_testu = [32, 64, 128, 256]

powtorzenia = 3       
epoki = 100           

wyniki_raport_batch = []
liczba_cech = X_train.shape[1]

print("="*60)
print(f"ROZPOCZYNAMY BADANIE: Wielkość paczki Sieć: [{n1}, {n2}], LR: {najlepszy_lr})")
print("="*60)

for b_size in batch_sizes_do_testu:
    print(f"\n---> Testuję Batch Size: {b_size:3} Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), b_size):
                end_idx = start_idx + b_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    
    wyniki_raport_batch.append({
        "Batch Size": b_size,
        "Śr. Czas 3 prób [s]": round(czas_trwania / powtorzenia, 1),
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2),
        "Najlepsze MAE (Test) [$]": round(np.min(mae_test_historia), 2)
    })

print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW ROZMIARU PACZKI (BATCH SIZE)")
print("="*60)
df_raport_batch = pd.DataFrame(wyniki_raport_batch)
print(df_raport_batch.to_string(index=False))

Badanie funkcji aktywacji

In [ ]:

# --- DEFINICJE NOWYCH FUNKCJI AKTYWACJI ---
class Activation_Sigmoid:
    def forward(self, inputs):
        self.inputs = inputs
        # Clipowanie zabezpiecza przed przepełnieniem matematycznym
        self.output = 1 / (1 + np.exp(-np.clip(inputs, -250, 250)))
    def backward(self, dvalues):
        self.dinputs = dvalues * (self.output * (1 - self.output))

class Activation_Tanh:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.tanh(inputs)
    def backward(self, dvalues):
        self.dinputs = dvalues * (1 - self.output ** 2)

class Activation_LeakyReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0.01 * inputs, inputs) # Puszcza 1% na ujemnych
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] *= 0.01


n1, n2 = 128, 64
najlepszy_lr = 0.05
najlepszy_batch = 128

aktywacje_do_testu = [
    ("ReLU", Activation_ReLU, Activation_ReLU),
    ("Leaky ReLU", Activation_LeakyReLU, Activation_LeakyReLU),
    ("Tanh", Activation_Tanh, Activation_Tanh),
    ("Sigmoid", Activation_Sigmoid, Activation_Sigmoid)
]

powtorzenia = 3       
epoki = 100           

wyniki_raport_act = []
liczba_cech = X_train.shape[1]

print("="*60)
print(f"ROZPOCZYNAMY BADANIE: Funkcja Aktywacji (Sieć: [128, 64], LR: 0.05, Batch: 128)")
print("="*60)

for nazwa, AktKlasa1, AktKlasa2 in aktywacje_do_testu:
    print(f"\n---> Testuję: {nazwa:10} (Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = AktKlasa1()  
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = AktKlasa2()  
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear() 
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    wyniki_raport_act.append({
        "Funkcja Aktywacji": nazwa,
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2)
    })


print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW FUNKCJI AKTYWACJI")
print("="*60)
df_raport_act = pd.DataFrame(wyniki_raport_act)
print(df_raport_act.to_string(index=False))

Badanie Epok

In [ ]:


n1, n2 = 128, 64
najlepszy_lr = 0.05
najlepszy_batch = 128

epoki_do_testu = [50, 100, 200, 300]
powtorzenia = 3       

wyniki_raport_epoki = []
liczba_cech = X_train.shape[1]

print("="*60)
print(f"ROZPOCZYNAMY BADANIE: Liczba Epok (Sieć: [128, 64], LR: 0.05, Batch: 128)")
print("="*60)

for e in epoki_do_testu:
    print(f"\n---> Testuję: {e:3} epok (Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU() 
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear() 
        
        loss_function = Loss_MSE()
        optimizer = Optimizer_SGD(learning_rate=najlepszy_lr) 
        
        
        for epoch in range(e): 
            for start_idx in range(0, len(X_train), najlepszy_batch):
                end_idx = start_idx + najlepszy_batch
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    wyniki_raport_epoki.append({
        "Liczba Epok": e,
        "Śr. Czas 3 prób [s]": round(czas_trwania / powtorzenia, 1),
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2)
    })


print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW LICZBY EPOK")
print("="*60)
df_raport_epoki = pd.DataFrame(wyniki_raport_epoki)
print(df_raport_epoki.to_string(index=False))

Finalna Sieć

In [ ]:


print("Trenowanie ostatecznego modelu z najlepszymi parametrami...")


n1, n2 = 128, 64
najlepszy_lr = 0.05
najlepszy_batch = 128
epoki = 100 


dense1 = Layer_Dense(X_train.shape[1], n1)
activation1 = Activation_ReLU()

dense2 = Layer_Dense(n1, n2)
activation2 = Activation_ReLU()

dense3 = Layer_Dense(n2, 1)
activation3 = Activation_Linear()

loss_function = Loss_MSE()
optimizer = Optimizer_SGD(learning_rate=najlepszy_lr)


for epoch in range(epoki):
    for start_idx in range(0, len(X_train), najlepszy_batch):
        end_idx = start_idx + najlepszy_batch
        X_batch = X_train[start_idx:end_idx]
        y_batch = y_train[start_idx:end_idx]
        
        dense1.forward(X_batch)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        loss_function.backward(activation3.output, y_batch)
        activation3.backward(loss_function.dinputs)
        dense3.backward(activation3.dinputs)
        activation2.backward(dense3.dinputs)
        dense2.backward(activation2.dinputs)
        activation1.backward(dense2.dinputs)
        dense1.backward(activation1.dinputs)
        
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.update_params(dense3)


dense1.forward(X_test)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
activation2.forward(dense2.output)
dense3.forward(activation2.output)
activation3.forward(dense3.output)

wymyslone_ceny_dolary = prawdziwa_cena(activation3.output)
prawdziwe_ceny_dolary = prawdziwa_cena(y_test)


mae = np.mean(np.abs(prawdziwe_ceny_dolary - wymyslone_ceny_dolary))
rmse = np.sqrt(np.mean((prawdziwe_ceny_dolary - wymyslone_ceny_dolary)**2))
ss_res = np.sum((prawdziwe_ceny_dolary - wymyslone_ceny_dolary)**2)
ss_tot = np.sum((prawdziwe_ceny_dolary - np.mean(prawdziwe_ceny_dolary))**2)
r2 = 1 - (ss_res / ss_tot)

non_zero = prawdziwe_ceny_dolary != 0
mape = np.mean(np.abs((prawdziwe_ceny_dolary[non_zero] - wymyslone_ceny_dolary[non_zero]) / prawdziwe_ceny_dolary[non_zero])) * 100

print("\n" + "="*50)
print("RAPORT KOŃCOWY: ZOPTYMALIZOWANA SIEĆ (DANE TESTOWE)")
print("="*50)
print(f"Przetwarzanie danych: Transformacja Logarytmiczna")
print(f"Architektura: [Wejście -> {n1} (ReLU) -> {n2} (ReLU) -> 1 (Linear)]")
print(f"Trening: LR = {najlepszy_lr}, Batch = {najlepszy_batch}, Epoki = {epoki}")
print("-" * 50)
print(f"1. MAE (Średnia pomyłka):     {mae:10.2f} $")
print(f"2. RMSE (Kara za ekstrema):   {rmse:10.2f} $")
print(f"3. Współczynnik R^2:          {r2:10.4f} (max 1.0)")
print(f"4. MAPE (Błąd procentowy):    {mape:10.2f} %")
print("="*50)

Trenowanie ostatecznego modelu z najlepszymi parametrami...

RAPORT KOŃCOWY: ZOPTYMALIZOWANA SIEĆ (DANE TESTOWE)
Przetwarzanie danych: Transformacja Logarytmiczna
Architektura: [Wejście -> 128 (ReLU) -> 64 (ReLU) -> 1 (Linear)]
Trening: LR = 0.05, Batch = 128, Epoki = 100
--------------------------------------------------
1. MAE (Średnia pomyłka):         108.89 $
2. RMSE (Kara za ekstrema):       329.03 $
3. Współczynnik R^2:              0.5106 (max 1.0)
4. MAPE (Błąd procentowy):        193.43 %
